# M2의 N 정의 4-arm 비교: Dunnhumby seed 42

A는 q_V-only(gate=1), B는 현재 반복거래율 N+q_V, C는 최신성 보정 활동 N+q_V, D는 BG/NBD의 향후 7일 기대거래횟수 N+q_V입니다. 네 arm은 N 입력 외에 3차원 고정 가치기저, rho=0.05, binary 2-layer LightGCN, K=5 uniform negative, plain BPR, 100 epoch가 모두 같습니다. B~D의 게이트는 절편 없이 같은 기울기 하나만 추천손실로 학습합니다.

기존 M1은 조건이 같은 완료 결과를 참고값으로만 재사용합니다. 이번 실험은 이미 노출된 DAY 684~690 개발구간의 단일 seed 기제 screen이므로 최종 test·holdout을 만들지 않으며, 이 결과 하나로 최종 후보를 확정하거나 유의성·일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'a2f32002706b45a749251875bf051d30c9f1a01f'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m2_n_proxy_value_basis_screen import (
    TRAINED_MODEL_IDS,
    configure_m2_n_proxy_screen,
    preflight_summary,
    run_m2_n_proxy_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
default_reference = Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_m2_m4_joint_historical_screen_v1/m5_m2_m4_joint_7cd818302fb1.json')
if not default_reference.exists():
    matches = list(Path('/content/drive/MyDrive').rglob('m5_m2_m4_joint_7cd818302fb1.json'))
    if len(matches) != 1:
        raise FileNotFoundError(
            '기존 M1 결과 7cd818302fb1을 현재 Drive에서 정확히 하나 찾을 수 없습니다. '
            f'검색 결과: {[str(path) for path in matches]}'
        )
    default_reference = matches[0]
    print('이동된 M1 결과를 찾았습니다:', default_reference)
cfg = configure_m2_n_proxy_screen(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m2_n_proxy_value_basis_development_screen_v1',
    m1_reference_json=str(default_reference),
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == list(TRAINED_MODEL_IDS)
assert summary['arms']['A'] == 'q_V-only, fixed gate=1'
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['m3_edge_weight'] is False
assert summary['fixed']['m4_loss_weight'] is False
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m2_n_proxy_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) 재사용 M1과 A~D M2 절대지표')
show(result_df)
print('2) M1 및 q_V-only 대비 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) ID 점수 대비 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) N 입력 진단')
show(result_df.attrs['n_proxy_diagnostics'])
print('5) BG/NBD 적합 진단')
print(json.dumps(result_df.attrs['bgnbd_diagnostics'], ensure_ascii=False, indent=2))
print('6) 게이트 작동 진단')
show(result_df.attrs['gate_diagnostics'])
print('7) 사전 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('8) 재사용 M1 출처')
print(json.dumps(result_df.attrs['reference_provenance'], ensure_ascii=False, indent=2))
print('9) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))